In [2]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

BOUNDARY_FILE = (
    ROOT / "data" / "processed" /
    "india_districts_simplified.geojson"
)

FORECAST_FILE = (
    ROOT / "data" / "processed" /
    "district_rainfall_forecast_20180727_20180731.csv"
)

forecast_date = pd.Timestamp("2018-07-29")

# Read district boundaries.
districts = gpd.read_file(BOUNDARY_FILE)

# Keep only permanent geographic columns to prevent merge suffixes.
districts = districts[
    ["shapeID", "shapeName", "geometry"]
].copy()

# Read the district forecast table.
forecast = pd.read_csv(
    FORECAST_FILE,
    parse_dates=["date"],
)

# Remove duplicate district-name column from the forecast table.
forecast = forecast.drop(
    columns=["shapeName"],
    errors="ignore",
)

# Select one forecast date.
selected_forecast = forecast.loc[
    forecast["date"].eq(forecast_date)
].copy()

# Merge forecast values with district geometry.
district_data = districts.merge(
    selected_forecast,
    on="shapeID",
    how="left",
    validate="one_to_one",
)

# Use representative-point latitude only for this regional diagnostic.
district_data["representative_latitude"] = (
    district_data.geometry.representative_point().y
)

# Approximate peninsular South India for the coverage test.
south_india = district_data.loc[
    district_data["representative_latitude"] < 17.0
].copy()

rainfall_columns = [
    "gefs_mean_mm",
    "global_corrected_mean_mm",
    "regime_corrected_mean_mm",
    "observed_mean_mm",
    "heavy_probability_max",
]

coverage_rows = []

for column in rainfall_columns:
    if column not in south_india.columns:
        coverage_rows.append(
            {
                "variable": column,
                "southern_districts": len(south_india),
                "available_values": 0,
                "missing_values": len(south_india),
                "zero_values": np.nan,
                "minimum": np.nan,
                "median": np.nan,
                "maximum": np.nan,
                "status": "Column not available",
            }
        )
        continue

    values = pd.to_numeric(
        south_india[column],
        errors="coerce",
    )

    coverage_rows.append(
        {
            "variable": column,
            "southern_districts": len(south_india),
            "available_values": int(values.notna().sum()),
            "missing_values": int(values.isna().sum()),
            "zero_values": int(values.eq(0).sum()),
            "minimum": values.min(),
            "median": values.median(),
            "maximum": values.max(),
            "status": "Available",
        }
    )

coverage_summary = pd.DataFrame(coverage_rows)

print("Selected date:", forecast_date.date())
print("All-India boundary districts:", len(district_data))
print("Forecast records for date:", len(selected_forecast))
print("Approximate South Indian districts:", len(south_india))
print("Merged columns:")
print(district_data.columns.tolist())

display(coverage_summary)

# Display only columns that actually exist.
display_columns = [
    "shapeName",
    "gefs_mean_mm",
    "global_corrected_mean_mm",
    "regime_corrected_mean_mm",
    "observed_mean_mm",
    "heavy_probability_max",
]

display_columns = [
    column
    for column in display_columns
    if column in south_india.columns
]

sort_column = (
    "regime_corrected_mean_mm"
    if "regime_corrected_mean_mm" in south_india.columns
    else "gefs_mean_mm"
)

south_table = south_india[display_columns].copy()

if sort_column in south_table.columns:
    south_table = south_table.sort_values(
        sort_column,
        ascending=False,
        na_position="last",
    )

display(south_table.head(40))

Selected date: 2018-07-29
All-India boundary districts: 735
Forecast records for date: 735
Approximate South Indian districts: 107
Merged columns:
['shapeID', 'shapeName', 'geometry', 'date', 'mapping_method', 'gefs_mean_mm', 'gefs_max_mm', 'observed_mean_mm', 'observed_max_mm', 'heavy_probability_mean', 'heavy_probability_max', 'very_heavy_probability_max', 'grid_cells', 'risk_level', 'very_heavy_experimental', 'representative_latitude']


,variable,southern_districts,available_values,missing_values,zero_values,minimum,median,maximum,status
0,gefs_mean_mm,107,107,0,5.0,0.000000,1.766667,48.20000,Available
1,global_corrected_mean_mm,107,0,107,NaN,NaN,NaN,NaN,Column not available
2,regime_corrected_mean_mm,107,0,107,NaN,NaN,NaN,NaN,Column not available
3,observed_mean_mm,107,107,0,10.0,0.000000,0.321667,19.50750,Available
4,heavy_probability_max,107,107,0,0.0,0.000135,0.000434,0.03873,Available


,shapeName,gefs_mean_mm,observed_mean_mm,heavy_probability_max
221,South Andaman,48.200000,4.381563,0.038730
220,North & Middle Andaman,20.612000,8.401250,0.022622
543,Ramanathapuram,20.453333,0.210417,0.002225
222,Nicobars,19.940000,19.507500,0.006118
366,Thrissur,16.340000,6.986250,0.004332
634,Thoothukkudi,13.861250,0.140000,0.005493
365,Kottayam,12.273335,8.835000,0.003802
545,Virudhunagar,12.218000,0.113000,0.004663
359,Alappuzha,11.660000,10.578750,0.002809
360,Ernakulam,10.252001,7.646500,0.005567


In [3]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

csv_files = sorted(
    (ROOT / "data" / "processed").rglob("*.csv")
)

search_results = []

for file in csv_files:
    try:
        columns = pd.read_csv(file, nrows=0).columns.tolist()

        search_results.append(
            {
                "file": file.name,
                "has_global_correction":
                    "global_corrected_mean_mm" in columns,
                "has_regime_correction":
                    "regime_corrected_mean_mm" in columns,
                "column_count": len(columns),
            }
        )
    except Exception as error:
        search_results.append(
            {
                "file": file.name,
                "has_global_correction": False,
                "has_regime_correction": False,
                "column_count": 0,
                "error": str(error),
            }
        )

search_table = pd.DataFrame(search_results)

display(
    search_table.sort_values(
        [
            "has_regime_correction",
            "has_global_correction",
        ],
        ascending=False,
    )
)

,file,has_global_correction,has_regime_correction,column_count
2,district_regime_corrected_forecast_20180727_20...,True,True,21
0,complete_grid_district_lookup.csv,False,False,5
1,district_rainfall_forecast_20180727_20180731.csv,False,False,14
3,gefs_grid_district_lookup.csv,False,False,4
4,heavy_rain_probability_metrics.csv,False,False,13
5,july2018_baseline_metrics.csv,False,False,6
6,july2018_extreme_metrics.csv,False,False,14
7,july2018_regime_correction_metrics.csv,False,False,8
8,july2018_regime_cv_predictions.csv,False,False,4
9,july2018_regime_labels.csv,False,False,9


In [4]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

SOURCE_FILE = (
    ROOT / "data" / "processed" /
    "district_regime_corrected_forecast.geojson"
)

OUTPUT_FILE = (
    ROOT / "data" / "processed" /
    "district_rainfall_forecast_20180727_20180731.csv"
)

print("Source exists:", SOURCE_FILE.exists())
print("Source:", SOURCE_FILE)

if not SOURCE_FILE.exists():
    raise FileNotFoundError(
        "The complete district GeoJSON was not found. "
        "Rerun notebook 20 to regenerate the deployment table."
    )

complete_gdf = gpd.read_file(SOURCE_FILE)

# Remove geometry because it is stored separately for deployment.
complete_table = pd.DataFrame(
    complete_gdf.drop(columns="geometry")
)

complete_table["date"] = pd.to_datetime(
    complete_table["date"]
).dt.strftime("%Y-%m-%d")

# Ensure one record exists for every district and date.
complete_table = (
    complete_table
    .drop_duplicates(
        subset=["shapeID", "date"],
        keep="first",
    )
    .sort_values(["date", "shapeName"])
    .reset_index(drop=True)
)

required_columns = [
    "shapeID",
    "date",
    "gefs_mean_mm",
    "global_corrected_mean_mm",
    "regime_corrected_mean_mm",
    "observed_mean_mm",
    "heavy_probability_max",
]

missing_columns = [
    column
    for column in required_columns
    if column not in complete_table.columns
]

if missing_columns:
    raise KeyError(
        f"Source GeoJSON is missing columns: {missing_columns}"
    )

complete_table.to_csv(
    OUTPUT_FILE,
    index=False,
)

print("Saved:", OUTPUT_FILE.exists())
print("Location:", OUTPUT_FILE)
print("Rows:", len(complete_table))
print("Districts:", complete_table["shapeID"].nunique())
print("Dates:", complete_table["date"].nunique())
print("Columns:")
print(complete_table.columns.tolist())

Source exists: True
Source: Z:\projects\monsoon-postprocessing\data\processed\district_regime_corrected_forecast.geojson
Saved: True
Location: Z:\projects\monsoon-postprocessing\data\processed\district_rainfall_forecast_20180727_20180731.csv
Rows: 3675
Districts: 735
Dates: 5
Columns:
['shapeID', 'shapeName', 'date', 'mapping_method', 'gefs_mean_mm', 'gefs_max_mm', 'observed_mean_mm', 'observed_max_mm', 'heavy_probability_mean', 'heavy_probability_max', 'very_heavy_probability_max', 'grid_cells', 'risk_level', 'very_heavy_experimental', 'correction_raw_mean_mm', 'global_corrected_mean_mm', 'global_corrected_max_mm', 'regime_corrected_mean_mm', 'regime_corrected_max_mm', 'regime_adjustment_mm', 'global_adjustment_mm']


In [5]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

FORECAST_FILE = (
    ROOT / "data" / "processed" /
    "district_rainfall_forecast_20180727_20180731.csv"
)

forecast = pd.read_csv(
    FORECAST_FILE,
    parse_dates=["date"],
)

required_columns = [
    "shapeID",
    "shapeName",
    "date",
    "gefs_mean_mm",
    "global_corrected_mean_mm",
    "regime_corrected_mean_mm",
    "observed_mean_mm",
    "heavy_probability_max",
]

validation = pd.DataFrame(
    {
        "column": required_columns,
        "available": [
            column in forecast.columns
            for column in required_columns
        ],
        "non_null_values": [
            (
                int(forecast[column].notna().sum())
                if column in forecast.columns
                else 0
            )
            for column in required_columns
        ],
    }
)

display(validation)

print("Rows:", len(forecast))
print("Districts:", forecast["shapeID"].nunique())
print("Dates:", forecast["date"].nunique())

,column,available,non_null_values
0,shapeID,True,3675
1,shapeName,True,3675
2,date,True,3675
3,gefs_mean_mm,True,3675
4,global_corrected_mean_mm,True,3675
5,regime_corrected_mean_mm,True,3675
6,observed_mean_mm,True,3675
7,heavy_probability_max,True,3675


Rows: 3675
Districts: 735
Dates: 5
